# MSPR ObRail — Étape 3b : Construction de la cible supervisée

**Membre 1 — Data Analyst / ML Lead**

Ce notebook **complète** `02_preparation.ipynb` (clustering) sans le modifier.
Il construit la **cible de classification** demandée par le livrable supervisé
(M2/M3 : régression logistique, RandomForest, XGBoost, LightGBM).

> ⚠️ **Cible synthétique et encadrée.** Le jeu de données ne contient que des
> services *ferroviaires* (aucune donnée avion). La cible est donc un **proxy**
> construit par règle de distance, pas une substitution observée. Deux garde-fous
> contre la *circularité* (cf. `docs/01_business.md` §8) :
> 1. on **exclut `distance_km`** des variables explicatives (c'est la source de la cible) ;
> 2. on **exclut `vitesse_kmh`** car elle vaut exactement `distance_km / duree * 60`
>    (corrélation 1.0) — la garder permettrait de recomposer la distance.
>
> Le modèle apprend donc à **retrouver la bande de distance à partir du profil
> d'exploitation** (durée, CO₂, horaire, nuit, transfrontalier, pays), ce qui est
> une tâche supervisée légitime et non circulaire.

## 1. Chargement du dataset de features

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42   # graine fixée -> reproductibilité

# Chargement de la sortie de l'étape 3 (notebook 02_preparation)
candidats = [Path("data/obrail_features.csv"),
             Path("../data/obrail_features.csv"),
             Path("obrail_features.csv")]
CSV = next((p for p in candidats if p.exists()), None)
if CSV is None:
    raise FileNotFoundError("Place obrail_features.csv dans data/ (sortie du notebook 02).")
ROOT = CSV.resolve().parent.parent

df = pd.read_csv(CSV, low_memory=False)
print(f"{len(df):,} services chargés depuis {CSV}")
print("Colonnes :", list(df.columns))

## 2. Définition de la cible synthétique `classe_substitution`

Trois classes fondées sur la distance (proxy de la compétitivité de l'aérien) :

| Classe | Règle | Lecture métier |
|---|---|---|
| `non_pertinent` | distance < 300 km | offre aérienne quasi absente, le train domine déjà |
| `substitution_difficile` | 300 ≤ distance < 500 km | zone de bascule, dépend de la vitesse du service |
| `substitution_possible` | distance ≥ 500 km | corridors où l'aérien court-courrier est présent : report modal = enjeu central |

Les **seuils sont justifiés** dans `docs/01_business.md` §8 (règle des ~3 h de
trajet, loi française 2023 sur les vols intérieurs, Green Deal). Les lignes dont
`distance_km` est manquante (≈ 55 %) sont laissées **non étiquetées** (`NaN`) et
sortent du périmètre supervisé.

In [ ]:
# --- Cible synthétique : bande de distance (proxy de compétitivité aérienne) ---
SEUIL_BAS, SEUIL_HAUT = 300, 500   # km  (seuils justifiés dans docs/01_business.md §8)

conditions = [
    df["distance_km"] < SEUIL_BAS,
    (df["distance_km"] >= SEUIL_BAS) & (df["distance_km"] < SEUIL_HAUT),
    df["distance_km"] >= SEUIL_HAUT,
]
labels = ["non_pertinent", "substitution_difficile", "substitution_possible"]

df["classe_substitution"] = np.select(conditions, labels, default=None)
# distance_km manquante -> non étiquetable -> NaN (hors périmètre supervisé)
df.loc[df["distance_km"].isna(), "classe_substitution"] = np.nan

ordre = ["non_pertinent", "substitution_difficile", "substitution_possible"]
df["classe_substitution"] = pd.Categorical(df["classe_substitution"],
                                            categories=ordre, ordered=True)

n_lab = int(df["classe_substitution"].notna().sum())
print(f"Lignes étiquetables (distance présente) : {n_lab:,} / {len(df):,} "
      f"({n_lab/len(df)*100:.1f} %)")
print("\nRépartition de la cible (sous-ensemble étiqueté) :")
vc = df["classe_substitution"].value_counts().reindex(ordre)
for k, v in vc.items():
    print(f"  {k:24s} {int(v):6,d}  ({v/n_lab*100:5.2f} %)")

## 3. Périmètre des variables explicatives (anti-circularité)

On fige ici la liste des features que M2 devra utiliser. `distance_km` et
`vitesse_kmh` en sont **volontairement exclues** (voir l'avertissement en tête de
notebook). `duree_minutes` et `emission_co2_kg` sont conservées : elles sont
*corrélées* à la distance (≈ 0.89) mais **pas déterministes** — ce sont de vraies
caractéristiques d'exploitation, c'est le signal légitime du modèle.

In [ ]:
# --- Périmètre des variables pour la classification (anti-circularité) ---
FEATURES_CLASSIF = [
    "duree_minutes",       # durée du service (corrélée mais NON déterministe vs distance)
    "emission_co2_kg",     # intensité carbone du service
    "heure_decimale",      # position horaire dans la journée
    "is_nuit",             # service de nuit (0/1)
    "is_transfrontalier",  # liaison transfrontalière (0/1)
    "code_pays_dep",       # pays de départ  (catégorielle -> one-hot côté M2)
    "code_pays_arr",       # pays d'arrivée  (catégorielle -> one-hot côté M2)
]

EXCLUES = {
    "distance_km": "source de la cible -> fuite directe (circularité)",
    "vitesse_kmh": "= distance_km / duree_minutes * 60 -> fuite déterministe (corr = 1.0)",
    "type_train": "redondante avec is_nuit (cohérence 100 %)",
    "duree_minutes_z / heure_decimale_z": "doublons standardisés (M2 standardise lui-même)",
    "id_trajet": "identifiant non prédictif",
}

print("Features retenues pour M2 :")
for f in FEATURES_CLASSIF:
    print("  +", f)
print("\nVariables volontairement EXCLUES :")
for k, v in EXCLUES.items():
    print(f"  - {k:38s} : {v}")

assert "distance_km" not in FEATURES_CLASSIF
assert "vitesse_kmh" not in FEATURES_CLASSIF
print("\nContrôle anti-circularité OK (ni distance_km ni vitesse_kmh dans les features).")

## 4. Découpage stratifié 70 / 15 / 15 (sous-ensemble étiqueté)

La colonne `split` existante (réservée au clustering de M1) **n'est pas modifiée**.
On crée une colonne dédiée `split_classif`, **stratifiée** sur la cible pour que la
classe minoritaire `substitution_possible` (~1 %) soit représentée dans train, val
et test.

In [ ]:
# --- Split stratifié 70/15/15 sur le sous-ensemble étiqueté ---
# La colonne `split` (clustering M1) n'est PAS touchée.
lab = df[df["classe_substitution"].notna()]
y = lab["classe_substitution"]

train_idx, temp_idx = train_test_split(
    lab.index, test_size=0.30, random_state=RANDOM_STATE, stratify=y)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=RANDOM_STATE, stratify=y.loc[temp_idx])

df["split_classif"] = "non_etiquete"
df.loc[train_idx, "split_classif"] = "train"
df.loc[val_idx,   "split_classif"] = "val"
df.loc[test_idx,  "split_classif"] = "test"

etiq = df.loc[df["split_classif"] != "non_etiquete", "split_classif"]
print("Répartition train/val/test (lignes étiquetées) :")
print((etiq.value_counts(normalize=True).mul(100).round(1)).astype(str) + " %")

print("\nReprésentation de chaque classe dans chaque split (stratification) :")
tab = pd.crosstab(df["split_classif"], df["classe_substitution"])
tab = tab.drop(index="non_etiquete", errors="ignore").reindex(["train", "val", "test"])
print(tab)
assert (tab > 0).all().all(), "Une classe est absente d'un split !"
print("\nOK : les 3 classes sont présentes dans train, val et test.")

## 5. Export — réécriture de `data/obrail_features.csv` (+2 colonnes)

In [ ]:
# --- Export : on réécrit obrail_features.csv en AJOUTANT 2 colonnes ---
cols_out = [
    "id_trajet",
    "duree_minutes", "emission_co2_kg", "distance_km", "vitesse_kmh",
    "heure_decimale", "is_nuit", "is_transfrontalier",
    "code_pays_dep", "code_pays_arr", "type_train",
    "duree_minutes_z", "heure_decimale_z",
    "split",                # split clustering (M1) - inchangé
    "classe_substitution",  # NOUVELLE cible supervisée (M2)
    "split_classif",        # NOUVEAU split stratifié pour la classification
]
out = df[cols_out].copy()
OUT = ROOT / "data" / "obrail_features.csv"
out.to_csv(OUT, index=False, encoding="utf-8")
print(f"Dataset mis à jour : {OUT}")
print(f"Dimensions : {out.shape[0]:,} lignes x {out.shape[1]} colonnes")
out[["id_trajet", "distance_km", "classe_substitution", "split_classif"]].head()

## 6. Contrat de livraison pour M2 / M3

- **Cible** : `classe_substitution` (3 classes ordonnées). Cible binaire dérivable
  trivialement (`substitution_possible` vs reste).
- **Features** : liste `FEATURES_CLASSIF` ci-dessus (one-hot sur `code_pays_*`,
  standardisation des numériques côté M2 ; les colonnes `*_z` fournies peuvent servir).
- **Split** : `split_classif` (`train` / `val` / `test`, stratifié).
- **Déséquilibre** : classe positive ~1 % → utiliser `class_weight='balanced'`
  (LogReg, RF), `scale_pos_weight` / poids d'échantillons (XGBoost, LightGBM), et
  **évaluer en macro-F1 / rappel par classe / matrice de confusion**, pas en accuracy.
- **Périmètre** : n'entraîner que sur `split_classif != 'non_etiquete'`.